# TN Config 1 – UHC Remittance Processor (v3)

## Key fixes over v2
- **Per-HC-line extraction with correct claim-block bounds**: one row per HC service line, with each row's Name/DOS/Procedure/Amount anchored to the *same* claim block. Eliminates the off-by-one drift in v2 where dates and names from neighboring claims bled across boundaries (e.g. Brenda Reid's $79.08 incorrectly showing 03/16/2026 instead of 03/24/2026).
- **OCR page-header stripping**: removes the repeating "Page N of M / UnitedHealthcare / column headers / Code" block that fragments claims across pages, so each claim's HC lines and Subtotal sit contiguously in the parsing text.
- **Patient-row aware block start**: walks back from each claim number to the start of its `TN##### NAME / POLICY` row, even when the patient header wraps onto a second line.
- **Per-claim subtotal validation**: each parsed block's HC-line sum is compared to the block's Subtotal "Paid to Provider"; mismatched blocks are routed to the LLM fallback automatically.
- **LLM fallback is per-claim, not whole-document**: only claims that fail deterministic parsing or subtotal reconciliation are sent to the LLM, with a focused OCR window for that single claim.
- **Reversal handling preserved**: negative HC amounts and Subtotals are extracted with their sign.
- **Filename total reconciliation**: end-of-run check against the dollar amount in the filename with a $1.00 threshold.

In [ ]:
# ── Cell 1: Install / restart ──────────────────────────────────────────────
%pip uninstall -y pandas numpy pandas-stubs pytz tzdata
%pip install "pandas==2.1.4" "numpy==1.26.4" "pytz<2025" "protobuf<5"

In [ ]:
dbutils.library.restartPython()

In [ ]:
# ── Cell 2: Imports & Azure OpenAI setup ──────────────────────────────────
import json
import re
import os
import pandas as pd
import numpy as np
import pdfplumber

from langchain.chat_models import AzureChatOpenAI
from langchain.schema import HumanMessage

api_key     = "3a46b22f850e4dd0bd2ed0586fb35a19"
api_base    = "https://bhsgenai.openai.azure.com/"
api_version = "2024-02-15-preview"

llm = AzureChatOpenAI(
    deployment_name="gpt-35-turbo-bsh",
    openai_api_base=api_base,
    openai_api_key=api_key,
    openai_api_type="azure",
    openai_api_version=api_version,
    temperature=0,
)

In [ ]:
# ── Cell 3: Load latest TN Config 1 PDF ───────────────────────────────────
inbound_dir = "/mnt/remittanceinbound/"

matching_files = [
    f for f in dbutils.fs.ls(inbound_dir)
    if f.name.startswith("TN Config 1") and f.name.endswith(".pdf")
]

if not matching_files:
    raise FileNotFoundError("No file starting with 'TN Config 1' and ending with '.pdf' found.")

latest_file       = sorted(matching_files, key=lambda x: x.modificationTime, reverse=True)[0]
original_filename = latest_file.name
pdf_path          = latest_file.path.replace("dbfs:", "/dbfs")
print(f"📄 Processing file: {pdf_path}")

# Parse expected total from filename  e.g. "... $112,300.19 ..."
m = re.search(r'\$([0-9,]+\.\d{2})', original_filename)
if not m:
    # Some filenames embed the total without a $ sign (e.g. "_126_752_06" for $126,752.06)
    m2 = re.search(r'_(\d{1,3}(?:_\d{3})*)_(\d{2})(?:_|$|\.)', original_filename)
    if m2:
        expected_total = float(m2.group(1).replace('_', '') + '.' + m2.group(2))
    else:
        raise ValueError(f"No dollar amount found in filename: {original_filename}")
else:
    expected_total = float(m.group(1).replace(",", ""))
print(f"💰 Expected total from filename: ${expected_total:,.2f}")

In [ ]:
# ── Cell 4: OCR both passes ────────────────────────────────────────────────
#
# Pass 1  – extract_text()        → compact / clean text
# Pass 2  – extract_text(layout=True) → preserves column spacing
#
# Pass-1 is the primary parsing input; Pass-2 is kept as a fallback when
# a claim's subtotal can't be reconciled from Pass-1.

ocr_lines_p1 = []
ocr_lines_p2 = []
page_stats   = []

with pdfplumber.open(pdf_path) as pdf:
    total_pages = len(pdf.pages)
    print(f"Total pages: {total_pages}")

    for page_num, page in enumerate(pdf.pages, start=1):
        # Pass 1 – plain text
        t1 = page.extract_text() or ""
        if not t1.strip():
            words = page.extract_words()
            t1 = " ".join(w["text"] for w in words) if words else ""
        t1 = t1.replace("\r", "\n")
        ocr_lines_p1.append(f"\n\n===== PAGE {page_num} =====\n")
        ocr_lines_p1.append(t1)

        # Pass 2 – layout text
        t2 = page.extract_text(layout=True) or ""
        if not t2.strip():
            t2 = t1
        t2 = t2.replace("\r", "\n")
        ocr_lines_p2.append(f"\n\n===== PAGE {page_num} =====\n")
        ocr_lines_p2.append(t2)

        page_stats.append((page_num, len(t1)))

ocr_text_p1 = "\n".join(ocr_lines_p1)
ocr_text_p2 = "\n".join(ocr_lines_p2)

missing_pages = [p for p, cnt in page_stats if cnt < 20]
print(f"Pages with little/no text: {missing_pages if missing_pages else 'None'}")
print(f"Usable pages: {total_pages - len(missing_pages)} / {total_pages}")

In [ ]:
# ── Cell 5: Clean OCR text + locate every claim block ────────────────────
#
# Strategy:
#   1. Remove every repeating PDF page header (the UnitedHealthcare /
#      "Payment Date / Payment Amount / column headers / Code" block)
#      that PDFPlumber prints once per page. These break claim blocks
#      across page boundaries and are the source of the off-by-one
#      drift in v2 (e.g. Brenda Reid's $79.08 showing 03/16/2026
#      instead of 03/24/2026: the parser was reading a previous claim's
#      date from before a page break).
#   2. Remove "Page N of M" footers.
#   3. Locate every claim number anchor (RC/RB/25-prefix).
#   4. For each claim anchor, walk back to the start of its patient
#      header line (the line starting with TN######). This becomes
#      block_start. block_end is the next claim's block_start.
#   5. Inside each block, extract:
#        - Account number, Patient name, Policy number (from TN##### row)
#        - Every HC service line  → one output row each
#        - Subtotal Paid to Provider for reconciliation


def clean_ocr(text: str) -> str:
    """Strip the repeating page-header noise from an OCR dump so claim
    blocks read as one continuous stream."""
    # Remove "===== PAGE N =====" through the end of the column headers
    # (which terminate on a line containing only "Code").
    cleaned = re.sub(
        r'={5,}\s*PAGE\s+\d+\s*={5,}.*?Code\n',
        '\n',
        text,
        flags=re.DOTALL,
    )
    # Remove standalone "Page N of M" lines
    cleaned = re.sub(r'Page\s+\d+\s+of\s+\d+\s*\n', '\n', cleaned)
    # Collapse multiple blank lines
    cleaned = re.sub(r'\n{2,}', '\n', cleaned)
    return cleaned


CLAIM_NUM_RE = re.compile(
    r'(?P<claim>(?:25[A-Z][A-Z0-9]{9}|RB\d{10}|RC\d{10}))'
)
PATIENT_HEADER_RE = re.compile(r'^TN\d{6,10}')
POLICY_CONT_RE    = re.compile(r'^\d{6,12}\b')


def find_claim_blocks(clean_text: str):
    """Return list of (block_start, block_end, claim_number, is_reversal)
    covering every claim in the cleaned OCR text."""
    # 1. Find all claim numbers
    matches = list(CLAIM_NUM_RE.finditer(clean_text))
    if not matches:
        return []

    block_starts = []
    for m in matches:
        pos = m.start()
        line_start = clean_text.rfind('\n', 0, pos) + 1
        check_start = line_start
        # Walk back up to 3 lines looking for a TN##### patient header.
        # If the line before is a policy/continuation row, keep walking.
        for _ in range(3):
            prev_nl = clean_text.rfind('\n', 0, check_start - 1)
            if prev_nl < 0:
                break
            prev_line = clean_text[prev_nl + 1:check_start - 1]
            if PATIENT_HEADER_RE.match(prev_line):
                check_start = prev_nl + 1
                break
            if POLICY_CONT_RE.match(prev_line):
                check_start = prev_nl + 1
                continue
            break
        block_starts.append(check_start)

    # 2. Pair each start with the next start as its end
    blocks = []
    for i, m in enumerate(matches):
        b_start = block_starts[i]
        b_end   = block_starts[i + 1] if i + 1 < len(block_starts) else len(clean_text)
        # Reversal detection: look near the claim number for "/Reversal"
        tail = clean_text[m.start():m.end() + 50]
        is_reversal = 'Reversal' in tail
        blocks.append((b_start, b_end, m.group('claim'), is_reversal))
    return blocks


# Build the cleaned text and block index once, reuse downstream
ocr_clean_p1 = clean_ocr(ocr_text_p1)
ocr_clean_p2 = clean_ocr(ocr_text_p2)

claim_blocks_p1 = find_claim_blocks(ocr_clean_p1)
print(f"Cleaned Pass-1 length: {len(ocr_clean_p1):,} chars")
print(f"Claim blocks found (Pass-1): {len(claim_blocks_p1)}")

In [ ]:
# ── Cell 6: Deterministic per-claim row extractor ────────────────────────
#
# For each claim block:
#   - Extract patient row fields:  TN##### NAME / POLICY
#   - Extract every HC service line as one output row
#   - Extract the Subtotal Paid-to-Provider for reconciliation
#
# Each HC line becomes a row with columns:
#   Policy Number, Name, AccountNumber, ClaimNbrType, DOS, Procedure Code,
#   Dollar Amount, Is Reversal, Reconciles (bool)
#
# A claim "reconciles" if the signed sum of its HC line amounts equals
# the Subtotal Paid-to-Provider for the block (within 1 cent). Blocks
# that fail reconciliation are flagged for LLM fallback.


# Patient header:  TN6113094  BRENDA J REID  /  116740717
NAME_RE = re.compile(
    r'(TN\d{6,10})\s+([A-Z][A-Z .\-\']+?)\s*/\s*(\d{6,12})'
)

# HC service line. Date may be alone on its line, then "HC:CODE $amount ..."
# on the next. Handle the optional trailing dash after the date.
HC_LINE_RE = re.compile(
    r'(\d{2}/\d{2}/\d{4})\s*-?\s*\n?\s*HC:([A-Z0-9]+(?::[A-Z0-9]+)?)\s+(-?\$[\d,]+\.\d{2})'
)

# Subtotal row:  Subtotal $79.08 $0.00 $0.00 $79.08 $0.00 $79.08 $0.00
# The 6th $-value is Paid to Provider.
SUBTOTAL_RE = re.compile(
    r'Subtotal\s+'
    r'(-?\$?[\d,]+\.\d{2})\s+'
    r'(-?\$?[\d,]+\.\d{2})\s+'
    r'(-?\$?[\d,]+\.\d{2})\s+'
    r'(-?\$?[\d,]+\.\d{2})\s+'
    r'(-?\$?[\d,]+\.\d{2})\s+'
    r'(-?\$?[\d,]+\.\d{2})'
)


def parse_money(s):
    try:
        return float(s.replace('$', '').replace(',', ''))
    except Exception:
        return None


def extract_block_rows(clean_text: str, b_start: int, b_end: int,
                       claim: str, is_reversal: bool):
    """Return (rows, paid_to_provider, reconciles_bool).

    rows is a list of dicts (one per HC line in the block).
    paid_to_provider is the block's Subtotal "Paid to Provider" value
    (None if no subtotal found).
    reconciles_bool is True if the HC-line sum matches paid_to_provider
    within 1 cent.
    """
    block = clean_text[b_start:b_end]

    # Patient header
    acct = name = policy = ''
    nm = NAME_RE.search(block)
    if nm:
        acct   = nm.group(1)
        name   = ' '.join(nm.group(2).split())
        policy = nm.group(3)

    # Subtotal
    paid_to_provider = None
    sm = SUBTOTAL_RE.search(block)
    if sm:
        paid_to_provider = parse_money(sm.group(6))

    # HC lines
    rows = []
    hc_sum = 0.0
    for hc in HC_LINE_RE.finditer(block):
        dos    = hc.group(1)
        proc   = hc.group(2)
        amount = parse_money(hc.group(3))
        if amount is None:
            continue
        hc_sum += amount
        rows.append({
            'Policy Number'  : policy,
            'Name'           : name,
            'AccountNumber'  : acct,
            'ClaimNbrType'   : claim,
            'DOS'            : dos,
            'Procedure Code' : proc,
            'Dollar Amount'  : amount,
            'Is Reversal'    : is_reversal,
        })

    reconciles = (
        paid_to_provider is not None
        and abs(hc_sum - paid_to_provider) < 0.01
    )
    return rows, paid_to_provider, reconciles


def run_deterministic(clean_text: str, blocks):
    """Run the deterministic parser. Returns (df, failed_claims) where
    failed_claims is a list of (claim_number, b_start, b_end, reason)
    for blocks that need LLM fallback."""
    all_rows = []
    failed   = []
    for b_start, b_end, claim, is_rev in blocks:
        rows, paid, ok = extract_block_rows(
            clean_text, b_start, b_end, claim, is_rev
        )
        if not rows:
            failed.append((claim, b_start, b_end, 'no HC lines'))
            continue
        if paid is None:
            failed.append((claim, b_start, b_end, 'no subtotal'))
            # Still keep the rows – we'll let LLM verify if needed
            all_rows.extend(rows)
            continue
        if not ok:
            failed.append((claim, b_start, b_end,
                           f'HC sum ${sum(r["Dollar Amount"] for r in rows):.2f} != Subtotal ${paid:.2f}'))
            # Still keep the rows; LLM will replace them if it returns
            # a better answer.
            all_rows.extend(rows)
            continue
        all_rows.extend(rows)
    return pd.DataFrame(all_rows), failed


print("Running deterministic extractor on Pass-1...")
df_det, failed_det = run_deterministic(ocr_clean_p1, claim_blocks_p1)
det_total = df_det['Dollar Amount'].astype(float).sum() if len(df_det) else 0.0
print(f"  Rows extracted   : {len(df_det)}")
print(f"  Claims found     : {df_det['ClaimNbrType'].nunique() if len(df_det) else 0}")
print(f"  Extracted total  : ${det_total:,.2f}")
print(f"  Expected total   : ${expected_total:,.2f}")
print(f"  Gap              : ${det_total - expected_total:,.2f}")
print(f"  Failed blocks    : {len(failed_det)}")
if failed_det:
    print("  First 5 failures:")
    for f in failed_det[:5]:
        print(f"    {f[0]} – {f[3]}")

In [ ]:
# ── Cell 7: LLM fallback per claim (only for failed blocks) ──────────────
#
# LLM is invoked only on claim blocks where the deterministic parser
# failed (no HC lines extracted, no Subtotal located, or HC-line sum
# disagrees with Subtotal). The block is sent to the LLM verbatim with
# a tight few-shot prompt.
#
# This keeps the LLM out of the happy path entirely on clean documents
# and concentrates compute on the small set of ambiguous blocks.

few_shot = """
You are extracting Medicaid remittance claim records from OCR text.
Return a JSON array (NEVER a single object) of HC service-line records.
Each record is one HC line from the claim and must contain:
  - ClientName     (patient name from the TN##### header row, e.g. "BRENDA J REID")
  - ClientNbr      (the 9-digit policy/subscriber ID)
  - AccountNumber  (the TN##### account number)
  - ClaimNbrType   (the RC/RB/25* claim number)
  - DOS            (the date of service on this HC line, MM/DD/YYYY)
  - ProcCode       (the HC procedure code, e.g. "T1019" or "T1019:UD")
  - AmountAllowed  (the "Paid to Provider" $-value on this HC line; keep sign)

CRITICAL RULES:
  • Produce ONE record per HC line in the block. Do not merge HC lines.
  • If the HC line amount is negative (reversal), keep the minus sign in AmountAllowed.
  • Do NOT use the Subtotal line as a record. Use only the HC lines.
  • The DOS for each record is the date that appears directly before "HC:" on that line, NOT the payment date or any header date.
  • Only output records present in the OCR block. Do not invent data.

Example OCR block:
TN5974192 GUADELUPE REYES/ 106760548
106351827 Previous Payment CHOICES 2
03/07/2026 -
HC:T1019 -$79.08 -- -- -$79.08 -- -$79.08 --
Subtotal -$79.08 $0.00 $0.00 -$79.08 $0.00 -$79.08 $0.00
RC1598447300/Reversal of Previous Payment

Example output:
[
  {
    "ClientName": "GUADELUPE REYES",
    "ClientNbr": "106760548",
    "AccountNumber": "TN5974192",
    "ClaimNbrType": "RC1598447300",
    "DOS": "03/07/2026",
    "ProcCode": "T1019",
    "AmountAllowed": "-79.08"
  }
]

Example OCR block with TWO HC dates (still ONE claim, TWO records):
TN5858228 CHRISTOPHER D CHRISTIANO/ 115766129 115766129 RC0827103500
02/08/2026 - HC:T1019:UD $217.47 -- -- $217.47 -- $217.47 --
02/18/2026 - HC:T1019 $79.08 -- -- $79.08 -- $79.08 --
Subtotal $296.55 $0.00 $0.00 $296.55 $0.00 $296.55 $0.00

Example output:
[
  {"ClientName":"CHRISTOPHER D CHRISTIANO","ClientNbr":"115766129","AccountNumber":"TN5858228","ClaimNbrType":"RC0827103500","DOS":"02/08/2026","ProcCode":"T1019:UD","AmountAllowed":"217.47"},
  {"ClientName":"CHRISTOPHER D CHRISTIANO","ClientNbr":"115766129","AccountNumber":"TN5858228","ClaimNbrType":"RC0827103500","DOS":"02/18/2026","ProcCode":"T1019","AmountAllowed":"79.08"}
]
"""


def call_llm_for_block(claim: str, block_text: str, fallback_block: str = None):
    """Send a block to the LLM, parse JSON, return a list of dicts.

    Retries once on JSON failure. If fallback_block is provided, retries
    once more with that. Returns [] on total failure.
    """
    def _ask(text):
        prompt = HumanMessage(
            content=f"{few_shot}\nNow extract claims from this OCR block:\n```\n{text}\n```"
        )
        resp = llm([prompt])
        raw = resp.content.strip()
        raw = re.sub(r'^```(?:json)?|```$', '', raw, flags=re.MULTILINE).strip()
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return [p for p in parsed if p.get('ClaimNbrType', '').replace('/COB','').replace('/Reversal','') == claim]

    for text_to_try in [block_text, fallback_block]:
        if text_to_try is None:
            continue
        for attempt in range(2):
            try:
                rows = _ask(text_to_try)
                if rows:
                    return rows
            except Exception:
                continue
    return []


def llm_rows_to_records(llm_rows, claim, is_reversal):
    """Convert LLM JSON entries to our row schema."""
    out = []
    for r in llm_rows:
        try:
            amount = float(str(r.get('AmountAllowed', 0)).replace('$','').replace(',',''))
        except Exception:
            amount = 0.0
        out.append({
            'Policy Number'  : str(r.get('ClientNbr', '')).strip(),
            'Name'           : ' '.join(str(r.get('ClientName', '')).split()),
            'AccountNumber'  : str(r.get('AccountNumber', '')).strip(),
            'ClaimNbrType'   : claim,
            'DOS'            : str(r.get('DOS', '')).strip(),
            'Procedure Code' : str(r.get('ProcCode', '')).strip(),
            'Dollar Amount'  : amount,
            'Is Reversal'    : is_reversal,
        })
    return out


# Only run if there were failed blocks
if failed_det:
    print(f"\nLLM fallback for {len(failed_det)} failed block(s)...")
    # Build a lookup of failed claims -> (b_start, b_end, reason)
    failed_lookup = {c: (s, e, r) for (c, s, e, r) in failed_det}

    # Also build a Pass-2 block index for the fallback
    claim_blocks_p2 = find_claim_blocks(ocr_clean_p2)
    p2_lookup = {c: (s, e) for (s, e, c, _) in claim_blocks_p2}

    llm_replacements = {}     # claim -> [rows]
    for claim, (b_start, b_end, reason) in failed_lookup.items():
        block_p1 = ocr_clean_p1[b_start:b_end]
        block_p2 = None
        if claim in p2_lookup:
            ps, pe = p2_lookup[claim]
            block_p2 = ocr_clean_p2[ps:pe]
        is_rev = 'Reversal' in block_p1
        llm_json = call_llm_for_block(claim, block_p1, block_p2)
        if llm_json:
            llm_replacements[claim] = llm_rows_to_records(llm_json, claim, is_rev)
            print(f"  ✅ {claim}: LLM returned {len(llm_json)} row(s)")
        else:
            print(f"  ❌ {claim}: LLM failed – keeping deterministic rows ({reason})")

    # Apply replacements: drop rows for failed claims and replace with LLM rows
    if llm_replacements:
        df_det = df_det[~df_det['ClaimNbrType'].isin(llm_replacements.keys())].copy()
        new_rows = []
        for rows in llm_replacements.values():
            new_rows.extend(rows)
        df_det = pd.concat([df_det, pd.DataFrame(new_rows)], ignore_index=True)
        print(f"  Applied LLM replacements for {len(llm_replacements)} claim(s).")
else:
    print("No failed blocks – skipping LLM fallback.")

post_llm_total = df_det['Dollar Amount'].astype(float).sum() if len(df_det) else 0.0
print(f"\nAfter LLM fallback:")
print(f"  Rows             : {len(df_det)}")
print(f"  Extracted total  : ${post_llm_total:,.2f}")
print(f"  Expected total   : ${expected_total:,.2f}")
print(f"  Gap              : ${post_llm_total - expected_total:,.2f}")

In [ ]:
# ── Cell 8: Post-process names and final columns ──────────────────────────

def shorten_name(name: str) -> str:
    """Reduce 'FIRST MIDDLE LAST' → 'FIRST LAST'. Also trims stray spaces."""
    parts = str(name).strip().split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[-1]}"
    return name


df_final = df_det.copy()
df_final["Name"] = df_final["Name"].fillna("").apply(shorten_name)

# Ensure Dollar Amount is numeric
df_final["Dollar Amount"] = pd.to_numeric(df_final["Dollar Amount"], errors="coerce").fillna(0.0)

# Reorder/keep only the output columns
output_cols = ["Policy Number", "Name", "DOS", "Procedure Code", "Dollar Amount"]
df_out = df_final[output_cols].copy()

final_total = df_out["Dollar Amount"].sum()
gap         = final_total - expected_total

print(f"\n{'='*55}")
print(f"  Extracted rows       : {len(df_out)}")
print(f"  Extracted total      : ${final_total:>12,.2f}")
print(f"  Expected total       : ${expected_total:>12,.2f}")
print(f"  Difference           : ${gap:>12,.2f}")
if abs(gap) < 1.0:
    print("  ✅ Totals match (within $1.00)")
else:
    print(f"  ⚠️  Gap > $1.00 – may reflect OCR read error on one claim amount.")
print(f"{'='*55}")

display(df_out.head(20))

In [ ]:
# ── Cell 9: Save to outbound ──────────────────────────────────────────────

outbound_dir    = "/dbfs/mnt/remittanceoutbound/"
output_filename = os.path.splitext(original_filename)[0] + "_results.csv"
output_path     = os.path.join(outbound_dir, output_filename)

df_out.to_csv(output_path, index=False)
print(f"Saved {len(df_out)} rows → {output_path}")

display(df_out)